# Customer Lifetime Value Estimation from Transaction Histories Under Right Censoring

**ASM II project | Online Retail II**

## Research objective

Estimate customer lifetime value (CLV) from transaction histories while explicitly accounting for the fact that a customer's true churn time may not be observed before the dataset ends.

The project treats customer churn as a **time-to-event outcome**. A customer who has not yet met the operational churn definition by the end of follow-up is right-censored, not automatically classified as a non-churner.

### Statistical methods used

1. Kaplan–Meier estimation of customer survival.
2. Hazard-function interpretation and comparison of survival curves.
3. Maximum-likelihood survival modelling with right-censored observations.
4. Cox proportional-hazards regression and proportional-hazards diagnostics.
5. Weibull accelerated-failure-time model as a parametric censored-likelihood comparison.
6. Regression methods for the monetary component: ordinary least squares, Ridge, LASSO and median/quantile regression.
7. Holdout evaluation and sensitivity analysis.

**Important design choice:** the original notebook's BG/NBD and Gamma-Gamma workflow is removed. Those models are not the central methods in the ASM II time-to-event syllabus. This notebook instead builds an explicit censoring-aware analysis.

## 1. Research design and operational definition

The Online Retail II data contain transaction dates and customer IDs, but do not directly record the date on which a customer permanently leaves the business. Therefore, churn must be operationally defined.

We use an **inactivity rule**: a customer is considered to have churned after `INACTIVITY_DAYS` consecutive days without a purchase. The default is 90 days; it is a modelling assumption, not a ground-truth label.

To avoid using future information as predictors, we use a landmark design:

- **Feature window:** transactions before a fixed landmark date.
- **Follow-up window:** transactions from the landmark date to the dataset end.
- Include customers who had purchased during the feature window and were still active at the landmark (their last pre-landmark purchase was within the inactivity threshold).
- Follow each customer's transaction dates during follow-up. The first gap exceeding the threshold defines an observed churn event at the previous purchase date plus the threshold.
- If no such gap is fully observed by the dataset end, the customer is right-censored.

This gives a common follow-up window and ensures the Cox model uses only pre-landmark predictors. The results estimate churn under this operational definition; they do not reveal a customer's unknowable permanent departure with certainty.

## 2. Install and import packages

Run the installation cell once in a fresh environment. The survival models use `lifelines`.

In [ ]:
%pip install -q lifelines openpyxl

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, QuantileRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullAFTFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
INACTIVITY_DAYS = 90
HOLDOUT_DAYS = 180
DISCOUNT_RATE_MONTHLY = 0.01

## 3. Load the Online Retail II data

Set `DATA_PATH` to the location of your downloaded Online Retail II file.

The loader accepts:
- an Excel workbook (`.xlsx` or `.xls`), optionally selecting a sheet;
- a CSV file (`.csv`).

The original notebook used the `Year 2010-2011` sheet. Here the sheet is configurable so that the analysis period is explicit.

In [ ]:
# EDIT THIS PATH to match your environment.
# Examples:
# DATA_PATH = "online_retail_II.xlsx"
# DATA_PATH = "/kaggle/input/online-retail-ii-uci/online_retail_II.xlsx"
# DATA_PATH = "online_retail_II.csv"

DATA_PATH = "online_retail_II.xlsx"
EXCEL_SHEET = "Year 2010-2011"

file_path = Path(DATA_PATH)
if not file_path.exists():
    raise FileNotFoundError(
        f"Cannot find {file_path.resolve()}. Download Online Retail II and update DATA_PATH."
    )

if file_path.suffix.lower() in [".xlsx", ".xls"]:
    raw = pd.read_excel(file_path, sheet_name=EXCEL_SHEET)
elif file_path.suffix.lower() == ".csv":
    raw = pd.read_csv(file_path, encoding="ISO-8859-1")
else:
    raise ValueError("Use an Excel (.xlsx/.xls) or CSV (.csv) file.")

print("Shape:", raw.shape)
display(raw.head())
print(raw.dtypes)

## 4. Data cleaning and transaction-level variables

We retain identified customers and positive sales transactions. Cancelled invoices and non-positive quantity/price records are excluded. We do not winsorize the transaction values by default: extreme purchases may be genuine customer behaviour. Outlier diagnostics are reported separately.

In [ ]:
df = raw.copy()

required = ["Invoice", "InvoiceDate", "Quantity", "Price", "Customer ID"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Customer ID"] = df["Customer ID"].astype("string").str.strip()

df = df.dropna(subset=["InvoiceDate", "Customer ID", "Invoice"])
df = df[~df["Invoice"].astype(str).str.startswith("C", na=False)]
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()

df["TotalPrice"] = df["Quantity"] * df["Price"]
df["Customer ID"] = df["Customer ID"].astype(str)

print("Cleaned shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Customers:", df["Customer ID"].nunique())
print("Invoices:", df["Invoice"].nunique())
display(df.head())

## 5. Exploratory data analysis

First inspect the distribution and concentration of customer spending and transaction frequency. These summaries help motivate transformations and robust regression later.

In [ ]:
customer_summary = df.groupby("Customer ID").agg(
    invoices=("Invoice", "nunique"),
    transactions=("Invoice", "size"),
    total_revenue=("TotalPrice", "sum"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
)
display(customer_summary[["invoices", "total_revenue"]].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
).T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(np.log1p(customer_summary["invoices"]), bins=40, ax=axes[0])
axes[0].set_title("Log(1 + number of invoices)")
sns.histplot(np.log1p(customer_summary["total_revenue"]), bins=40, ax=axes[1])
axes[1].set_title("Log(1 + total revenue)")
plt.tight_layout()
plt.show()

## 6. Define the landmark and construct right-censored outcomes

The landmark is set `HOLDOUT_DAYS` before the final transaction date. Features are computed strictly before the landmark; churn outcomes and future revenue are computed after it.

A customer is eligible if:
1. they made at least one purchase before the landmark; and
2. their last pre-landmark purchase was no more than `INACTIVITY_DAYS` before the landmark.

For each eligible customer, the first post-landmark inter-purchase gap longer than the inactivity threshold is an event. Otherwise, the observation is censored at the dataset end. This procedure uses the transaction sequence, rather than treating every customer with no recent purchase as a confirmed churner.

In [ ]:
observation_end = df["InvoiceDate"].max().normalize()
landmark = observation_end - pd.Timedelta(days=HOLDOUT_DAYS)

feature_tx = df[df["InvoiceDate"] < landmark].copy()
followup_tx = df[df["InvoiceDate"] >= landmark].copy()

# Features available at the landmark (strictly pre-landmark).
features = feature_tx.groupby("Customer ID").agg(
    first_purchase=("InvoiceDate", "min"),
    last_pre_landmark=("InvoiceDate", "max"),
    invoice_count=("Invoice", "nunique"),
    total_revenue_pre=("TotalPrice", "sum"),
    avg_invoice_value=("TotalPrice", lambda x: x.groupby(feature_tx.loc[x.index, "Invoice"]).sum().mean()),
    unique_products=("StockCode", "nunique") if "StockCode" in feature_tx.columns else ("Invoice", "nunique"),
    country=("Country", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown") if "Country" in feature_tx.columns else ("Invoice", lambda x: "Unknown")
)

features["pre_landmark_recency_days"] = (
    landmark - features["last_pre_landmark"].dt.normalize()
).dt.days

eligible = features[features["pre_landmark_recency_days"] <= INACTIVITY_DAYS].copy()
eligible = eligible[eligible["invoice_count"] >= 1].copy()

# Customer-level follow-up outcome construction.
future_dates = (
    followup_tx.groupby("Customer ID")["InvoiceDate"]
    .apply(lambda s: sorted(pd.to_datetime(s).dt.normalize().unique()))
    .to_dict()
)
future_revenue = followup_tx.groupby("Customer ID")["TotalPrice"].sum()

outcomes = []
for customer_id, row in eligible.iterrows():
    dates = future_dates.get(customer_id, [])
    dates = [pd.Timestamp(d) for d in dates]

    # Include the last known purchase before landmark as the start of the first gap.
    previous_purchase = row["last_pre_landmark"].normalize()
    event_date = None
    for purchase_date in dates:
        if (purchase_date - previous_purchase).days > INACTIVITY_DAYS:
            event_date = previous_purchase + pd.Timedelta(days=INACTIVITY_DAYS)
            break
        previous_purchase = purchase_date

    # If no long gap was observed between purchases, test the final gap to study end.
    if event_date is None:
        if (observation_end - previous_purchase).days >= INACTIVITY_DAYS:
            event_date = previous_purchase + pd.Timedelta(days=INACTIVITY_DAYS)

    event_observed = int(event_date is not None and event_date <= observation_end)
    stop_date = event_date if event_observed else observation_end
    duration_days = max((stop_date - landmark).days, 1)
    active_days = max(min(duration_days, HOLDOUT_DAYS), 1)

    outcomes.append({
        "Customer ID": customer_id,
        "duration_days": duration_days,
        "event_observed": event_observed,
        "future_revenue": float(future_revenue.get(customer_id, 0.0)),
        "active_days": active_days
    })

outcomes = pd.DataFrame(outcomes).set_index("Customer ID")
analysis = eligible.join(outcomes, how="inner")
analysis["future_monthly_revenue"] = (
    analysis["future_revenue"] / (analysis["active_days"] / 30.4375)
)

print("Landmark:", landmark)
print("Observation end:", observation_end)
print("Eligible customers:", len(analysis))
print("Observed churn events:", int(analysis["event_observed"].sum()))
print("Right-censored customers:", int((analysis["event_observed"] == 0).sum()))
print("Censoring proportion:", 1 - analysis["event_observed"].mean())
display(analysis.head())

## 7. Inspect censoring and follow-up

`event_observed = 1` means the operational inactivity-based churn event was observed. `event_observed = 0` means the customer remained under observation without a confirmed event by the end of follow-up.

The censoring indicator is essential: a censored customer must contribute survival information up to their censoring time, but must not be treated as if their churn time were known.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=analysis, x="event_observed", ax=axes[0])
axes[0].set_xticks([0, 1], ["Censored", "Churn observed"])
axes[0].set_title("Event indicator")
sns.histplot(data=analysis, x="duration_days", hue="event_observed",
             bins=30, element="step", stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Observed follow-up time")
plt.tight_layout()
plt.show()

## 8. Kaplan–Meier estimator

For event time \(T\), the survival function is \(S(t)=P(T>t)\). The Kaplan–Meier estimator is

\[
\widehat S(t)=\prod_{t_j\leq t}\left(1-\frac{d_j}{n_j}\right),
\]

where \(d_j\) is the number of observed events at time \(t_j\), and \(n_j\) is the number at risk immediately before that time.

Customers who are right-censored leave the risk set at their censoring time; they are not counted as churn events.

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(
    durations=analysis["duration_days"],
    event_observed=analysis["event_observed"],
    label="All eligible customers"
)
ax = kmf.plot_survival_function(ci_show=True, figsize=(9, 5))
ax.set_xlabel("Days since landmark")
ax.set_ylabel("Estimated probability of not yet churning")
ax.set_title("Kaplan–Meier customer survival estimate")
plt.show()

display(kmf.survival_function_.head(10))
print("Median survival time:", kmf.median_survival_time_)

## 9. Compare survival across customer groups

We form groups using **pre-landmark invoice frequency**. The log-rank test compares survival distributions between two groups. This is an unadjusted comparison; it does not control for other customer characteristics.

In [ ]:
analysis["frequency_group"] = pd.qcut(
    analysis["invoice_count"].rank(method="first"),
    q=2,
    labels=["Lower frequency", "Higher frequency"]
)

fig, ax = plt.subplots(figsize=(9, 5))
for group, subset in analysis.groupby("frequency_group", observed=True):
    KaplanMeierFitter().fit(
        subset["duration_days"], subset["event_observed"], label=str(group)
    ).plot_survival_function(ax=ax, ci_show=True)
ax.set_title("Kaplan–Meier curves by pre-landmark purchase frequency")
ax.set_xlabel("Days since landmark")
ax.set_ylabel("Survival probability")
plt.show()

low = analysis[analysis["frequency_group"] == "Lower frequency"]
high = analysis[analysis["frequency_group"] == "Higher frequency"]
lr = logrank_test(
    low["duration_days"], high["duration_days"],
    event_observed_A=low["event_observed"],
    event_observed_B=high["event_observed"]
)
print("Log-rank test statistic:", lr.test_statistic)
print("Log-rank p-value:", lr.p_value)

## 10. Prepare predictors for survival regression

The Cox model is fitted using customer-level characteristics measured before the landmark. Skewed monetary and count variables are log-transformed. We exclude customer ID and the outcome variables from the predictors.

The Cox proportional-hazards model is

\[
h(t\mid x)=h_0(t)\exp(x^T\beta).
\]

A positive coefficient corresponds to a higher hazard (higher instantaneous churn rate), conditional on the other included variables. The hazard ratio for a one-unit increase in a predictor is \(\exp(\beta_j)\).

In [ ]:
survival_data = analysis[[
    "duration_days", "event_observed",
    "invoice_count", "total_revenue_pre", "avg_invoice_value",
    "unique_products", "pre_landmark_recency_days"
]].copy()

survival_data["log_invoice_count"] = np.log1p(survival_data["invoice_count"])
survival_data["log_total_revenue"] = np.log1p(survival_data["total_revenue_pre"])
survival_data["log_avg_invoice_value"] = np.log1p(survival_data["avg_invoice_value"])
survival_data["log_unique_products"] = np.log1p(survival_data["unique_products"])

model_columns = [
    "duration_days", "event_observed",
    "log_invoice_count", "log_total_revenue",
    "log_avg_invoice_value", "log_unique_products",
    "pre_landmark_recency_days"
]
survival_data = survival_data[model_columns].replace([np.inf, -np.inf], np.nan).dropna()

train_surv, test_surv = train_test_split(
    survival_data, test_size=0.25, random_state=RANDOM_STATE,
    stratify=survival_data["event_observed"]
)
print("Training observations:", len(train_surv))
print("Test observations:", len(test_surv))

## 11. Cox proportional-hazards model

The Cox partial likelihood estimates regression coefficients without specifying the baseline hazard function. Right-censored observations contribute to the risk sets until their censoring times.

We fit on the training set and evaluate concordance on held-out customers. Concordance measures how well the model ranks relative event risk; it is not a measure of calibrated CLV accuracy.

In [ ]:
cox = CoxPHFitter(penalizer=0.01)
cox.fit(
    train_surv,
    duration_col="duration_days",
    event_col="event_observed"
)
display(cox.summary[["coef", "exp(coef)", "se(coef)", "p", "exp(coef) lower 95%", "exp(coef) upper 95%"]])

test_risk = cox.predict_partial_hazard(test_surv)
test_cindex = concordance_index(
    test_surv["duration_days"],
    -test_risk,
    test_surv["event_observed"]
)
print("Test concordance index:", round(test_cindex, 4))

## 12. Proportional-hazards assumption

The Cox model assumes hazard ratios are constant over time. Use the Schoenfeld-residual-based diagnostic provided by `lifelines`. A small p-value for a predictor may indicate a violation; inspect plots and consider whether a time-varying effect or stratification is warranted.

Do not interpret a diagnostic p-value mechanically, especially with a large sample.

In [ ]:
cox.check_assumptions(train_surv, p_value_threshold=0.05, show_plots=True)

## 13. Parametric censored-likelihood model: Weibull AFT

As a parametric comparison, fit a Weibull accelerated-failure-time model. Its likelihood uses the event density for observed churn events and the survival probability for right-censored customers:

\[
L(\theta)=\prod_{i:\delta_i=1} f(t_i\mid x_i;\theta)
\prod_{i:\delta_i=0} S(t_i\mid x_i;\theta).
\]

This is the key reason censoring must be represented explicitly. A censored observation contributes \(S(t_i)\), not \(f(t_i)\).

The Weibull AFT model is a parametric extension for comparison with the semi-parametric Cox model.

In [ ]:
aft = WeibullAFTFitter()
aft.fit(
    train_surv,
    duration_col="duration_days",
    event_col="event_observed"
)
display(aft.summary[["coef", "exp(coef)", "se(coef)", "p"]])
print("Cox partial AIC:", cox.AIC_partial_)
print("Weibull AFT AIC:", aft.AIC_)

## 14. Monetary-value modelling using ASM II regression methods

The survival model estimates the probability that a customer remains active. We separately estimate the customer's monthly revenue rate using transactions after the landmark.

The response is future revenue divided by observed active exposure (in months). The predictors are restricted to pre-landmark information. We compare:
- ordinary least squares;
- Ridge regression;
- LASSO regression;
- median (0.5 quantile) regression, a robust alternative to squared-error fitting.

Because revenue is highly skewed and may contain zeros, the models predict `log(1 + future monthly revenue)`. Predictions are transformed back using `expm1` and truncated at zero. Model performance is compared on the log scale using a held-out test set.

In [ ]:
regression_data = analysis[[
    "future_monthly_revenue",
    "invoice_count", "total_revenue_pre", "avg_invoice_value",
    "unique_products", "pre_landmark_recency_days"
]].copy()

regression_data["target_log_monthly_revenue"] = np.log1p(
    regression_data["future_monthly_revenue"].clip(lower=0)
)
for col in ["invoice_count", "total_revenue_pre", "avg_invoice_value", "unique_products"]:
    regression_data["log_" + col] = np.log1p(regression_data[col].clip(lower=0))

predictors = [
    "log_invoice_count", "log_total_revenue_pre",
    "log_avg_invoice_value", "log_unique_products",
    "pre_landmark_recency_days"
]
regression_data = regression_data.replace([np.inf, -np.inf], np.nan).dropna()

X = regression_data[predictors]
y = regression_data["target_log_monthly_revenue"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

regression_models = {
    "OLS": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25))),
    "LASSO": make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=20000)),
    "Median quantile": make_pipeline(StandardScaler(), QuantileRegressor(
        quantile=0.5, alpha=0.01, solver="highs"
    ))
}

regression_results = []
fitted_regressors = {}
for name, model in regression_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted_regressors[name] = model
    regression_results.append({
        "Model": name,
        "MAE (log scale)": mean_absolute_error(y_test, pred),
        "RMSE (log scale)": mean_squared_error(y_test, pred) ** 0.5,
        "R-squared (log scale)": r2_score(y_test, pred)
    })

regression_results = pd.DataFrame(regression_results).sort_values("RMSE (log scale)")
display(regression_results)

## 15. Regression diagnostics and coefficient interpretation

Inspect residual behaviour for OLS and the selected regularized model. The project should discuss skewness, heteroscedasticity, influential observations and the limitations of interpreting coefficients after regularization.

The quantile model estimates a conditional median rather than a conditional mean. It is therefore a robustness comparison, not a direct replacement for the mean-revenue prediction used in expected-value calculations.

In [ ]:
ols = fitted_regressors["OLS"]
ols_pred = ols.predict(X_test)
residuals = y_test - ols_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ols_pred, residuals, alpha=0.35)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Fitted log monthly revenue")
axes[0].set_ylabel("Residual")
axes[0].set_title("OLS residuals vs fitted")
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title("OLS residual distribution")
plt.tight_layout()
plt.show()

## 16. Combine survival and monetary value into a CLV estimate

For customer \(i\), a practical finite-horizon expected-value estimate is

\[
\widehat{CLV}_i(H)=
\widehat r_i
\sum_{m=1}^{H}
\frac{\widehat S_i(30.4375m)}
{(1+d)^m},
\]

where:
- \(\widehat r_i\) is predicted monthly revenue while active;
- \(\widehat S_i(t)\) is the Cox-model survival probability at time \(t\);
- \(d\) is the monthly discount rate;
- \(H\) is the forecast horizon in months.

This is a simplified revenue-based CLV, not profit-based CLV: costs, gross margins, acquisition costs and future changes in spending are not available in the present specification. It assumes the predicted monthly revenue rate is earned while the customer remains active.

In [ ]:
# Use the best-performing regression model by held-out RMSE.
best_name = regression_results.iloc[0]["Model"]
best_regressor = fitted_regressors[best_name]
print("Selected monetary model:", best_name)

# Refit on all available customer observations for final descriptive estimates.
best_regressor.fit(X, y)
predicted_monthly_revenue = np.maximum(
    np.expm1(best_regressor.predict(X)), 0
)

# Cox survival curves for the same customer feature rows.
clv_surv_features = survival_data.drop(columns=["duration_days", "event_observed"])
# Align regression observations with survival observations by customer index.
common_ids = X.index.intersection(clv_surv_features.index)
X_common = X.loc[common_ids]
surv_common = clv_surv_features.loc[common_ids]

monthly_times = np.arange(1, 13) * 30.4375
survival_curves = cox.predict_survival_function(surv_common, times=monthly_times)

monthly_revenue_common = pd.Series(
    np.maximum(np.expm1(best_regressor.predict(X_common)), 0),
    index=common_ids
)

discount_factors = (1 + DISCOUNT_RATE_MONTHLY) ** np.arange(1, 13)
discounted_survival_sum = (
    survival_curves.to_numpy().T / discount_factors
).sum(axis=1)

clv_estimates = pd.DataFrame(index=common_ids)
clv_estimates["predicted_monthly_revenue"] = monthly_revenue_common
clv_estimates["12m_survival_adjusted_clv"] = (
    clv_estimates["predicted_monthly_revenue"] * discounted_survival_sum
)
clv_estimates = clv_estimates.sort_values("12m_survival_adjusted_clv", ascending=False)

display(clv_estimates.head(10))
print("Total estimated 12-month CLV:", clv_estimates["12m_survival_adjusted_clv"].sum())

## 17. Customer-level survival and CLV visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(clv_estimates["predicted_monthly_revenue"], bins=40, ax=axes[0])
axes[0].set_title("Predicted monthly revenue while active")
sns.histplot(clv_estimates["12m_survival_adjusted_clv"], bins=40, ax=axes[1])
axes[1].set_title("12-month survival-adjusted CLV")
plt.tight_layout()
plt.show()

display(clv_estimates.describe().T)

## 18. Sensitivity analysis: churn definition

The inactivity threshold is a substantive assumption. Repeat the customer-level outcome construction for alternative thresholds (for example, 60, 90 and 120 days) and compare:
- number of eligible customers;
- observed churn proportion;
- Kaplan–Meier median survival;
- Cox concordance.

A result that changes substantially with the threshold should be reported as sensitive to the operational definition of churn.

In [ ]:
def construct_survival_outcomes(threshold_days):
    rows = []
    for customer_id, row in features.iterrows():
        if row["pre_landmark_recency_days"] > threshold_days:
            continue

        dates = future_dates.get(customer_id, [])
        dates = [pd.Timestamp(d) for d in dates]
        previous = row["last_pre_landmark"].normalize()
        event_date = None

        for date in dates:
            if (date - previous).days > threshold_days:
                event_date = previous + pd.Timedelta(days=threshold_days)
                break
            previous = date

        if event_date is None and (observation_end - previous).days >= threshold_days:
            event_date = previous + pd.Timedelta(days=threshold_days)

        event = int(event_date is not None and event_date <= observation_end)
        stop = event_date if event else observation_end
        rows.append((customer_id, max((stop - landmark).days, 1), event))

    return pd.DataFrame(rows, columns=["Customer ID", "duration_days", "event_observed"]).set_index("Customer ID")

sensitivity_rows = []
for threshold in [60, 90, 120]:
    temp = construct_survival_outcomes(threshold)
    if len(temp) == 0 or temp["event_observed"].sum() == 0:
        continue
    km_temp = KaplanMeierFitter().fit(
        temp["duration_days"], temp["event_observed"]
    )
    sensitivity_rows.append({
        "Inactivity threshold": threshold,
        "Eligible customers": len(temp),
        "Observed events": int(temp["event_observed"].sum()),
        "Censoring proportion": 1 - temp["event_observed"].mean(),
        "KM median survival (days)": km_temp.median_survival_time_
    })

display(pd.DataFrame(sensitivity_rows))

## 19. Why some ASM II methods are not forced into the model

The syllabus also covers the EM algorithm for missing-data likelihoods and the MM algorithm for optimization. They are not automatically appropriate just because they appear in the course:

- **EM:** missing customer IDs cannot be reliably imputed because the identity needed to join transactions is unknown. Other missing descriptive fields are not required for the core survival predictors. If a relevant numeric predictor has meaningful missingness, an EM-based model could be considered, but it should be justified by the missing-data mechanism and assumptions.
- **MM:** the Cox partial-likelihood and the regression estimators here can be fitted using their standard optimization routines. An MM implementation would be an additional algorithmic comparison, not a necessary part of the substantive CLV question.

This is deliberate: the project should apply course methods that answer the research question, rather than insert every syllabus topic without a statistical reason.


## 20. Limitations and interpretation

1. **Churn is not directly observed.** The 90-day inactivity rule is a proxy. Customers may return after long gaps.
2. **Right-censoring assumption.** The analysis treats censoring at the dataset end as non-informative conditional on the modelled predictors. This assumption should be discussed.
3. **Single landmark.** The model is conditional on customers being active at the landmark; it does not describe customers who had already become inactive before it.
4. **One observation window.** A different year or landmark may produce different results.
5. **Revenue is not profit.** The CLV estimate does not subtract costs or incorporate margins.
6. **Cox proportional-hazards assumption.** Violations can make constant hazard-ratio interpretations inappropriate.
7. **Prediction uncertainty.** The displayed CLV is a point estimate. A stronger final report should quantify uncertainty, for example by customer-level bootstrap that repeats the complete modelling pipeline.

## 21. What to report to the professor

Your report should prioritise the statistical question, not just the final customer ranking:

- Explain why transaction histories create right-censored time-to-event observations.
- State the inactivity-based operational definition and landmark design.
- Show the Kaplan–Meier survival curve and explain the risk set.
- Interpret Cox hazard ratios with confidence intervals and report PH diagnostics.
- Compare Cox and Weibull AFT models using appropriate fit and predictive summaries.
- Compare OLS, Ridge, LASSO and median regression for the monetary component.
- Explain the CLV combination formula and its assumptions.
- Include the inactivity-threshold sensitivity analysis.

**Do not claim that the model observes the true customer lifetime.** It estimates survival under a stated churn definition, using incomplete transaction histories.